In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from .text_utils import chunk_text, normalize_text, split_sentences, tokenize, top_k


@dataclass
class MatchEvidence:
    sentence_a: str
    sentence_b: str
    similarity: float
    fuzzy_ratio: float


@dataclass
class ComparisonResult:
    file_a: str
    file_b: str
    cosine_similarity: float
    char_cosine_similarity: float
    fuzzy_similarity: float
    combined_score: float
    is_flagged: bool
    top_evidence: List[MatchEvidence] = field(default_factory=list)

    def to_dict(self) -> Dict:
        data = asdict(self)
        data["top_evidence"] = [asdict(x) for x in self.top_evidence]
        return data


@dataclass
class PlagiarismReport:
    threshold: float
    files: List[str]
    results: List[ComparisonResult]

    def to_dict(self) -> Dict:
        return {
            "threshold": self.threshold,
            "files": self.files,
            "results": [r.to_dict() for r in self.results],
        }


class PlagiarismChecker:
    """NLP pipeline for similarity detection and plagiarism flagging."""

    def __init__(
        self,
        threshold: float = 0.72,
        word_ngram_range: Tuple[int, int] = (1, 2),
        char_ngram_range: Tuple[int, int] = (3, 5),
        max_features: int = 20_000,
    ) -> None:
        self.threshold = threshold
        self.word_vectorizer = TfidfVectorizer(
            preprocessor=normalize_text,
            lowercase=False,
            token_pattern=r"(?u)\b\w+\b",
            ngram_range=word_ngram_range,
            max_features=max_features,
        )
        self.char_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            preprocessor=normalize_text,
            lowercase=False,
            ngram_range=char_ngram_range,
            max_features=max_features,
        )

    def load_documents(self, paths: Sequence[Path | str]) -> Dict[str, str]:
        documents: Dict[str, str] = {}
        for path in paths:
            p = Path(path)
            if p.is_dir():
                for child in sorted(p.glob("*.txt")):
                    documents[child.name] = child.read_text(encoding="utf-8", errors="ignore")
            else:
                documents[p.name] = p.read_text(encoding="utf-8", errors="ignore")
        return documents

    def fit_transform(self, documents: Dict[str, str]):
        names = list(documents.keys())
        raw_texts = [documents[n] for n in names]
        word_matrix = self.word_vectorizer.fit_transform(raw_texts)
        char_matrix = self.char_vectorizer.fit_transform(raw_texts)
        return names, raw_texts, word_matrix, char_matrix

    def _sentence_similarity_matrix(self, sentences_a: List[str], sentences_b: List[str]) -> List[MatchEvidence]:
        if not sentences_a or not sentences_b:
            return []

        combined = sentences_a + sentences_b
        tfidf = TfidfVectorizer(
            preprocessor=normalize_text,
            lowercase=False,
            token_pattern=r"(?u)\b\w+\b",
            ngram_range=(1, 2),
            max_features=10_000,
        )
        try:
            matrix = tfidf.fit_transform(combined)
            a_mat = matrix[: len(sentences_a)]
            b_mat = matrix[len(sentences_a) :]
            cos = cosine_similarity(a_mat, b_mat)
        except ValueError:
            cos = np.zeros((len(sentences_a), len(sentences_b)))

        candidates: List[Tuple[float, int, int]] = []
        for i, sa in enumerate(sentences_a):
            for j, sb in enumerate(sentences_b):
                fuzzy = fuzz.token_set_ratio(sa, sb) / 100.0
                score = 0.7 * float(cos[i, j]) + 0.3 * fuzzy
                candidates.append((score, i, j))

        best = top_k(candidates, 3)
        evidence: List[MatchEvidence] = []
        for score, i, j in best:
            sa = sentences_a[i]
            sb = sentences_b[j]
            evidence.append(
                MatchEvidence(
                    sentence_a=sa,
                    sentence_b=sb,
                    similarity=round(float(score), 4),
                    fuzzy_ratio=round(fuzz.token_set_ratio(sa, sb) / 100.0, 4),
                )
            )
        return evidence

    def compare_pair(
        self,
        file_a: str,
        file_b: str,
        raw_a: str,
        raw_b: str,
        word_matrix_a,
        word_matrix_b,
        char_matrix_a,
        char_matrix_b,
    ) -> ComparisonResult:
        cosine_score = float(cosine_similarity(word_matrix_a, word_matrix_b)[0, 0])
        char_score = float(cosine_similarity(char_matrix_a, char_matrix_b)[0, 0])
        fuzzy_score = fuzz.token_set_ratio(normalize_text(raw_a), normalize_text(raw_b)) / 100.0

        combined = 0.50 * cosine_score + 0.25 * char_score + 0.25 * fuzzy_score
        flagged = combined >= self.threshold

        sentences_a = split_sentences(raw_a)
        sentences_b = split_sentences(raw_b)
        evidence = self._sentence_similarity_matrix(sentences_a, sentences_b)

        return ComparisonResult(
            file_a=file_a,
            file_b=file_b,
            cosine_similarity=round(cosine_score, 4),
            char_cosine_similarity=round(char_score, 4),
            fuzzy_similarity=round(fuzzy_score, 4),
            combined_score=round(combined, 4),
            is_flagged=flagged,
            top_evidence=evidence,
        )

    def compare_documents(self, documents: Dict[str, str]) -> PlagiarismReport:
        if len(documents) < 2:
            raise ValueError("At least two documents are required for comparison.")

        names, raw_texts, word_matrix, char_matrix = self.fit_transform(documents)
        results: List[ComparisonResult] = []

        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                result = self.compare_pair(
                    names[i],
                    names[j],
                    raw_texts[i],
                    raw_texts[j],
                    word_matrix[i],
                    word_matrix[j],
                    char_matrix[i],
                    char_matrix[j],
                )
                results.append(result)

        results.sort(key=lambda r: r.combined_score, reverse=True)
        return PlagiarismReport(threshold=self.threshold, files=names, results=results)

    def compare_paths(self, paths: Sequence[Path | str]) -> PlagiarismReport:
        documents = self.load_documents(paths)
        return self.compare_documents(documents)

    def save_json_report(self, report: PlagiarismReport, output_path: Path | str) -> Path:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(json.dumps(report.to_dict(), indent=2), encoding="utf-8")
        return output_path

    def save_html_report(self, report: PlagiarismReport, output_path: Path | str) -> Path:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        rows = []
        for r in report.results:
            evidence_html = "".join(
                f"<details><summary>Evidence {idx + 1}</summary>"
                f"<p><b>A:</b> {escape_html(ev.sentence_a)}</p>"
                f"<p><b>B:</b> {escape_html(ev.sentence_b)}</p>"
                f"<p><b>Similarity:</b> {ev.similarity:.4f} | <b>Fuzzy:</b> {ev.fuzzy_ratio:.4f}</p>"
                f"</details>"
                for idx, ev in enumerate(r.top_evidence)
            )
            rows.append(
                f"<tr class={'flagged' if r.is_flagged else 'clean'}>"
                f"<td>{escape_html(r.file_a)}</td>"
                f"<td>{escape_html(r.file_b)}</td>"
                f"<td>{r.cosine_similarity:.4f}</td>"
                f"<td>{r.char_cosine_similarity:.4f}</td>"
                f"<td>{r.fuzzy_similarity:.4f}</td>"
                f"<td><b>{r.combined_score:.4f}</b></td>"
                f"<td>{'Yes' if r.is_flagged else 'No'}</td>"
                f"<td>{evidence_html}</td>"
                f"</tr>"
            )

        html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Plagiarism Report</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; color: #111827; }}
    .meta {{ margin-bottom: 18px; padding: 12px 16px; background: #f3f4f6; border-radius: 12px; }}
    table {{ width: 100%; border-collapse: collapse; }}
    th, td {{ border: 1px solid #d1d5db; padding: 10px; vertical-align: top; }}
    th {{ background: #111827; color: white; }}
    tr.flagged {{ background: #fef2f2; }}
    tr.clean {{ background: #f9fafb; }}
    details {{ margin-top: 6px; }}
    summary {{ cursor: pointer; font-weight: 600; }}
  </style>
</head>
<body>
  <h1>Plagiarism Analysis Report</h1>
  <div class="meta">
    <div><b>Threshold:</b> {report.threshold:.2f}</div>
    <div><b>Files:</b> {len(report.files)}</div>
    <div><b>Comparisons:</b> {len(report.results)}</div>
  </div>
  <table>
    <thead>
      <tr>
        <th>File A</th>
        <th>File B</th>
        <th>Word Cosine</th>
        <th>Char Cosine</th>
        <th>Fuzzy</th>
        <th>Combined</th>
        <th>Flagged</th>
        <th>Evidence</th>
      </tr>
    </thead>
    <tbody>
      {''.join(rows)}
    </tbody>
  </table>
</body>
</html>
"""
        output_path.write_text(html, encoding="utf-8")
        return output_path


def escape_html(text: str) -> str:
    return (
        text.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )
